# ⚡ Notebook 5: Application Caching with Redis

Caching is the most effective way to scale reads. Redis sits between your application and database, serving frequently accessed data from memory.

## Learning Objectives

By the end of this notebook, you'll understand:
- How application-level caching works
- Cache-aside pattern
- TTL strategies
- Basic cache invalidation

---

### 🔍 Open RedisInsight to Watch Cache Operations!

1. Go to **http://localhost:5540**
2. Click "Add Redis Database" → Host: `redis`, Port: `6379`
3. Open the **Browser** tab to see keys as they're created!

## 🛠️ Setup

**1. Start PostgreSQL + Redis + visualization tools** (from the lab root):

```bash
cd 04-patterns/scaling-reads
docker compose up -d
uv sync
```

**2. Select the `.venv` kernel**

Click the kernel picker in the top-right of this notebook and choose the
`.venv` Python interpreter. If it doesn't appear, reload the VS Code window
(`Cmd+Shift+P` → "Developer: Reload Window") and try again.

### 🔍 Visualization tools (optional but recommended)

| Tool | URL | Use it to |
|------|-----|-----------|
| Adminer (PostgreSQL) | http://localhost:8080 | See tables, run SQL, view execution plans |
| RedisInsight | http://localhost:5540 | Watch cache keys, TTLs, hits/misses |

**Adminer login**: System `PostgreSQL`, Server `postgres`, User `demo`,
Password `demo`, Database `scaling_demo`.


In [1]:
import redis
import psycopg2
import json
import time
from typing import Optional

DB_CONFIG = {
    "host": "localhost",
    "port": 5432,
    "database": "scaling_demo",
    "user": "demo",
    "password": "demo"
}

r = redis.Redis(host='localhost', port=6379, decode_responses=True)

def get_db_connection():
    return psycopg2.connect(**DB_CONFIG)

try:
    r.ping()
    print("✅ Redis connected")
except:
    print("❌ Redis not running. Start with: docker compose up -d")

try:
    conn = get_db_connection()
    print("✅ PostgreSQL connected")
    conn.close()
except Exception as e:
    print(f"❌ PostgreSQL failed: {e}")

✅ Redis connected
✅ PostgreSQL connected


## 🏗️ Cache-Aside Pattern

The most common caching pattern: check cache first, fall back to database.

In [2]:
print("🏗️ Cache-Aside Pattern")
print("=" * 60)
print("""
READ PATH:
─────────────────────────────────────────────────────────────

    Application
        │
        ├──► 1. Check Cache
        │         │
        │    ┌────┴────┐
        │    │         │
        │  HIT ✅    MISS ❌
        │    │         │
        │    │    2. Query Database
        │    │         │
        │    │    3. Store in Cache
        │    │         │
        └────┴─────────┘
                │
           Return Data

─────────────────────────────────────────────────────────────

• Cache HIT:  ~1ms  (memory read)
• Cache MISS: ~10ms (DB query + cache write)
""")

🏗️ Cache-Aside Pattern

READ PATH:
─────────────────────────────────────────────────────────────

    Application
        │
        ├──► 1. Check Cache
        │         │
        │    ┌────┴────┐
        │    │         │
        │  HIT ✅    MISS ❌
        │    │         │
        │    │    2. Query Database
        │    │         │
        │    │    3. Store in Cache
        │    │         │
        └────┴─────────┘
                │
           Return Data

─────────────────────────────────────────────────────────────

• Cache HIT:  ~1ms  (memory read)
• Cache MISS: ~10ms (DB query + cache write)



In [3]:
class CachedUserRepository:
    def __init__(self, redis_client, ttl_seconds: int = 300):
        self.redis = redis_client
        self.ttl = ttl_seconds
        self.stats = {"hits": 0, "misses": 0}
    
    def _cache_key(self, user_id: int) -> str:
        return f"user:{user_id}"
    
    def get_user(self, user_id: int) -> Optional[dict]:
        cache_key = self._cache_key(user_id)
        
        cached = self.redis.get(cache_key)
        if cached:
            self.stats["hits"] += 1
            return json.loads(cached)
        
        self.stats["misses"] += 1
        
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "SELECT id, username, email, display_name, follower_count "
            "FROM users WHERE id = %s", (user_id,)
        )
        row = cursor.fetchone()
        conn.close()
        
        if not row:
            return None
        
        user = {
            "id": row[0],
            "username": row[1],
            "email": row[2],
            "display_name": row[3],
            "follower_count": row[4]
        }
        
        self.redis.setex(cache_key, self.ttl, json.dumps(user))
        
        return user
    
    def invalidate(self, user_id: int):
        self.redis.delete(self._cache_key(user_id))

repo = CachedUserRepository(r, ttl_seconds=60)
print("✅ CachedUserRepository created with 60s TTL")

✅ CachedUserRepository created with 60s TTL


In [4]:
print("🔬 Testing Cache-Aside Pattern")
print("=" * 60)

repo.invalidate(1)

print("\n📖 First read (cache MISS):")
start = time.time()
user = repo.get_user(1)
elapsed = (time.time() - start) * 1000
print(f"   Time: {elapsed:.2f}ms")
print(f"   User: {user['username']}")
print(f"   Stats: {repo.stats}")

print("\n📖 Second read (cache HIT):")
start = time.time()
user = repo.get_user(1)
elapsed = (time.time() - start) * 1000
print(f"   Time: {elapsed:.2f}ms")
print(f"   User: {user['username']}")
print(f"   Stats: {repo.stats}")

print("\n💡 Second read was MUCH faster - came from cache!")
print("   👀 Check RedisInsight for 'user:1' key!")

🔬 Testing Cache-Aside Pattern

📖 First read (cache MISS):
   Time: 11.71ms
   User: user1
   Stats: {'hits': 0, 'misses': 1}

📖 Second read (cache HIT):
   Time: 0.32ms
   User: user1
   Stats: {'hits': 1, 'misses': 1}

💡 Second read was MUCH faster - came from cache!
   👀 Check RedisInsight for 'user:1' key!


## ⏰ TTL Strategies

In [5]:
print("⏰ TTL (Time-To-Live) Strategies")
print("=" * 60)
print("""
TTL determines how long cached data lives before expiring.

SHORT TTL (5-60 seconds)
─────────────────────────────────────────────────────────────
• Use for: Frequently changing data
• Examples: Stock prices, live scores, inventory counts
• Trade-off: More cache misses, but fresher data

MEDIUM TTL (5-15 minutes)
─────────────────────────────────────────────────────────────
• Use for: Semi-static data
• Examples: User profiles, product details, blog posts
• Trade-off: Good balance of freshness and hit rate

LONG TTL (1 hour - 24 hours)
─────────────────────────────────────────────────────────────
• Use for: Rarely changing data
• Examples: Config settings, category lists, translations
• Trade-off: High hit rate, but may serve stale data

NO TTL (Never expires)
─────────────────────────────────────────────────────────────
• Use for: Immutable data
• Examples: URL shortener mappings, historical data
• Trade-off: Must invalidate manually on changes
""")

⏰ TTL (Time-To-Live) Strategies

TTL determines how long cached data lives before expiring.

SHORT TTL (5-60 seconds)
─────────────────────────────────────────────────────────────
• Use for: Frequently changing data
• Examples: Stock prices, live scores, inventory counts
• Trade-off: More cache misses, but fresher data

MEDIUM TTL (5-15 minutes)
─────────────────────────────────────────────────────────────
• Use for: Semi-static data
• Examples: User profiles, product details, blog posts
• Trade-off: Good balance of freshness and hit rate

LONG TTL (1 hour - 24 hours)
─────────────────────────────────────────────────────────────
• Use for: Rarely changing data
• Examples: Config settings, category lists, translations
• Trade-off: High hit rate, but may serve stale data

NO TTL (Never expires)
─────────────────────────────────────────────────────────────
• Use for: Immutable data
• Examples: URL shortener mappings, historical data
• Trade-off: Must invalidate manually on changes



In [6]:
print("🔬 Demonstrating TTL Expiration")
print("=" * 60)

r.setex("demo:short_ttl", 3, "I expire in 3 seconds")
print("\n✏️ Set key with 3 second TTL")

print("\n⏱️ Watching TTL countdown:")
for i in range(5):
    value = r.get("demo:short_ttl")
    ttl = r.ttl("demo:short_ttl")
    status = "✅" if value else "❌ EXPIRED"
    print(f"   Second {i}: TTL={ttl}s - {status}")
    time.sleep(1)

print("\n💡 After TTL expires, next read causes cache miss!")

🔬 Demonstrating TTL Expiration

✏️ Set key with 3 second TTL

⏱️ Watching TTL countdown:
   Second 0: TTL=3s - ✅


   Second 1: TTL=2s - ✅


   Second 2: TTL=1s - ✅


   Second 3: TTL=-2s - ❌ EXPIRED


   Second 4: TTL=-2s - ❌ EXPIRED



💡 After TTL expires, next read causes cache miss!


## 🔄 Cache Invalidation Strategies

In [7]:
print("🔄 Cache Invalidation Strategies")
print("=" * 60)
print("""
1. TIME-BASED (TTL)
─────────────────────────────────────────────────────────────
• Simplest approach - data expires after fixed time
• Pro: No extra code needed
• Con: May serve stale data until TTL expires

2. WRITE-THROUGH
─────────────────────────────────────────────────────────────
• Update cache immediately when writing to DB
• Pro: Cache always has latest data
• Con: Adds latency to writes

3. WRITE-BEHIND (INVALIDATE)
─────────────────────────────────────────────────────────────
• Delete cache entry on write, let next read populate
• Pro: Simple, eventual consistency
• Con: Next read has cache miss

4. EVENT-DRIVEN
─────────────────────────────────────────────────────────────
• Publish event on write, subscriber invalidates
• Pro: Decoupled, works across services
• Con: More complex infrastructure
""")

🔄 Cache Invalidation Strategies

1. TIME-BASED (TTL)
─────────────────────────────────────────────────────────────
• Simplest approach - data expires after fixed time
• Pro: No extra code needed
• Con: May serve stale data until TTL expires

2. WRITE-THROUGH
─────────────────────────────────────────────────────────────
• Update cache immediately when writing to DB
• Pro: Cache always has latest data
• Con: Adds latency to writes

3. WRITE-BEHIND (INVALIDATE)
─────────────────────────────────────────────────────────────
• Delete cache entry on write, let next read populate
• Pro: Simple, eventual consistency
• Con: Next read has cache miss

4. EVENT-DRIVEN
─────────────────────────────────────────────────────────────
• Publish event on write, subscriber invalidates
• Pro: Decoupled, works across services
• Con: More complex infrastructure



In [8]:
class CachedUserRepositoryWithInvalidation(CachedUserRepository):
    def update_user(self, user_id: int, display_name: str):
        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "UPDATE users SET display_name = %s WHERE id = %s",
            (display_name, user_id)
        )
        conn.commit()
        conn.close()
        
        self.invalidate(user_id)
        print(f"   🗑️ Invalidated cache for user:{user_id}")

repo2 = CachedUserRepositoryWithInvalidation(r, ttl_seconds=300)

print("🔬 Write-Behind Invalidation Demo")
print("=" * 60)

print("\n1. Read user (populates cache):")
user = repo2.get_user(1)
print(f"   Display name: {user['display_name']}")

print("\n2. Update user (invalidates cache):")
repo2.update_user(1, "New Display Name")

print("\n3. Read again (cache miss, gets fresh data):")
user = repo2.get_user(1)
print(f"   Display name: {user['display_name']}")
print(f"   Stats: {repo2.stats}")

🔬 Write-Behind Invalidation Demo

1. Read user (populates cache):
   Display name: New Display Name

2. Update user (invalidates cache):
   🗑️ Invalidated cache for user:1

3. Read again (cache miss, gets fresh data):
   Display name: New Display Name
   Stats: {'hits': 1, 'misses': 1}


## 📊 Cache Hit Rate

In [9]:
import random

def simulate_traffic(repo, num_requests: int, popular_users: list, all_users: range):
    for user_id in popular_users:
        repo.invalidate(user_id)
    repo.stats = {"hits": 0, "misses": 0}
    
    for _ in range(num_requests):
        if random.random() < 0.8:
            user_id = random.choice(popular_users)
        else:
            user_id = random.choice(list(all_users))
        repo.get_user(user_id)
    
    total = repo.stats["hits"] + repo.stats["misses"]
    hit_rate = (repo.stats["hits"] / total) * 100 if total > 0 else 0
    return hit_rate, repo.stats

print("📊 Cache Hit Rate Analysis")
print("=" * 60)

repo3 = CachedUserRepository(r, ttl_seconds=60)

popular = [1, 2, 3, 4, 5]
all_users = range(1, 101)

hit_rate, stats = simulate_traffic(repo3, 1000, popular, all_users)

print(f"\n📊 Results after 1000 requests:")
print(f"   Hits: {stats['hits']}")
print(f"   Misses: {stats['misses']}")
print(f"   Hit Rate: {hit_rate:.1f}%")

print("\n💡 Real-world hit rates:")
print("   • 80%+ is good for user data")
print("   • 95%+ for static content")
print("   • >99% for CDN-cached assets")

📊 Cache Hit Rate Analysis



📊 Results after 1000 requests:
   Hits: 988
   Misses: 12
   Hit Rate: 98.8%

💡 Real-world hit rates:
   • 80%+ is good for user data
   • 95%+ for static content
   • >99% for CDN-cached assets


## 🧪 Quick Quiz

1. **What's the cache-aside pattern?**

2. **When would you use a short TTL vs long TTL?**

3. **What's write-behind invalidation?**

## 🚫 Negative Caching (cache the "not found" too)

A common miss: something doesn't exist in the database, so we never cache it,
so every lookup hits the DB. Attackers can exploit this to DOS your database
by requesting random non-existent IDs.

**Fix**: cache the *absence* of the row too, with a short TTL.

```
Request: get_user(99999999)
  Cache miss → DB returns None → cache {"__null__": true} for 30s
Next 1000 requests for user 99999999:
  Cache hit → return None. Zero DB queries.
```


In [10]:
# Demo: negative caching
class CachedUserRepositoryWithNegCache(CachedUserRepository):
    NULL_MARKER = "__NULL__"
    NEG_TTL = 30  # shorter TTL for negatives — the user may be created soon

    def get_user(self, user_id: int):
        cache_key = self._cache_key(user_id)
        cached = self.redis.get(cache_key)

        if cached == self.NULL_MARKER:
            self.stats["hits"] += 1
            return None
        if cached:
            self.stats["hits"] += 1
            return json.loads(cached)

        self.stats["misses"] += 1

        conn = get_db_connection()
        cursor = conn.cursor()
        cursor.execute(
            "SELECT id, username, email FROM users WHERE id = %s", (user_id,)
        )
        row = cursor.fetchone()
        conn.close()

        if not row:
            # Cache the "not found" to protect the DB from repeated misses
            self.redis.setex(cache_key, self.NEG_TTL, self.NULL_MARKER)
            return None

        user = {"id": row[0], "username": row[1], "email": row[2]}
        self.redis.setex(cache_key, self.ttl, json.dumps(user))
        return user


neg_repo = CachedUserRepositoryWithNegCache(r, ttl_seconds=60)

# Clean slate
for uid in [99999991, 99999992]:
    r.delete(f"user:{uid}")

print("First lookup of a user that doesn't exist (cache miss → DB):")
neg_repo.get_user(99999991)
print(f"   Stats: {neg_repo.stats}")

print("\nNext 5 lookups of that same missing user (should all be hits):")
for _ in range(5):
    neg_repo.get_user(99999991)
print(f"   Stats: {neg_repo.stats}")
print("\n💡 Only 1 DB query for 6 requests — even though the user doesn't exist!")


First lookup of a user that doesn't exist (cache miss → DB):
   Stats: {'hits': 0, 'misses': 1}

Next 5 lookups of that same missing user (should all be hits):
   Stats: {'hits': 5, 'misses': 1}

💡 Only 1 DB query for 6 requests — even though the user doesn't exist!


## 🔥 Cache Warming (pre-fill the cache before traffic hits)

When your cache is empty (after a deploy, failover, or maintenance), every
first request is a miss. On a busy site that can cause a **cold-cache
stampede** — the DB gets crushed right when you restart.

**Fix**: warm the cache *before* serving traffic by pre-loading known-hot keys.

Common warming strategies:
- On startup: load the top-N most-requested items into Redis
- On cache-invalidation: write the fresh value back instead of just deleting
- Periodically: a background job keeps popular keys refreshed


In [11]:
# Demo: warm the cache with the top-N most-followed users before traffic starts

def warm_user_cache(repo, top_n: int = 20):
    """Pre-populate Redis with the most popular users (by follower count)."""
    conn = get_db_connection()
    cursor = conn.cursor()
    cursor.execute(
        "SELECT id FROM users ORDER BY follower_count DESC LIMIT %s",
        (top_n,),
    )
    hot_ids = [row[0] for row in cursor.fetchall()]
    conn.close()

    for uid in hot_ids:
        repo.invalidate(uid)   # ensure fresh
        repo.get_user(uid)     # populates cache
    return hot_ids


warm_repo = CachedUserRepository(r, ttl_seconds=300)
warm_repo.stats = {"hits": 0, "misses": 0}

print("Warming cache with top-20 users...")
warmed_ids = warm_user_cache(warm_repo, top_n=20)
print(f"   Warmed {len(warmed_ids)} users; stats after warm-up: {warm_repo.stats}")

print("\nNow simulate real traffic — mostly hits:")
warm_repo.stats = {"hits": 0, "misses": 0}
for _ in range(200):
    warm_repo.get_user(random.choice(warmed_ids))
print(f"   Stats: {warm_repo.stats}")
hit_rate = warm_repo.stats["hits"] / sum(warm_repo.stats.values()) * 100
print(f"   Hit rate: {hit_rate:.1f}% (no cold-start penalty!)")


Warming cache with top-20 users...


   Warmed 20 users; stats after warm-up: {'hits': 0, 'misses': 20}

Now simulate real traffic — mostly hits:


   Stats: {'hits': 200, 'misses': 0}
   Hit rate: 100.0% (no cold-start penalty!)


In [12]:
print("📝 Quiz Answers")
print("=" * 50)
print()
print("1. Cache-aside pattern:")
print("   - Check cache first")
print("   - On miss: query DB, store in cache")
print("   - Application controls caching logic")
print()
print("2. TTL choices:")
print("   Short (seconds): Frequently changing data")
print("   Long (hours): Rarely changing data")
print("   Based on staleness tolerance!")
print()
print("3. Write-behind invalidation:")
print("   - Delete cache entry after DB write")
print("   - Next read will have cache miss")
print("   - Simple but causes one extra DB query")

📝 Quiz Answers

1. Cache-aside pattern:
   - Check cache first
   - On miss: query DB, store in cache
   - Application controls caching logic

2. TTL choices:
   Short (seconds): Frequently changing data
   Long (hours): Rarely changing data
   Based on staleness tolerance!

3. Write-behind invalidation:
   - Delete cache entry after DB write
   - Next read will have cache miss
   - Simple but causes one extra DB query


## 📚 Summary

### Key Takeaways

1. **Cache-aside is the default pattern** - check cache, fallback to DB
2. **TTL based on staleness tolerance** - how stale can data be?
3. **Invalidate on writes** - delete or update cache entries
4. **Hit rate matters** - 80%+ is good, 95%+ is great
5. **Popular data stays cached** - Zipf's law helps us!

### Next Up

In **Notebook 6**, we'll cover advanced cache patterns:
- Cache stampede prevention
- Hot key problem
- Cache versioning